## 7.2 Implementation

### 7.2.1 Neural Network as Feature Extractor

---

### **Solution Outline**

1. Train the best MLP obtained earlier.
2. Extract penultimate layer features:
   
   $$
   \phi_{NN}(x) = h^{(L-1)}(x)
   $$

3. Visualize features using t-SNE.
4. Define neural kernel:

   $$
   k(x_i, x_j) = \langle \phi_{NN}(x_i), \phi_{NN}(x_j) \rangle
   $$

5. Train Kernel SVR using this neural kernel on Phase 4 data.

---

### **Mathematical Formulation**

Given an MLP:

$$
h^{(l)}(x) = \sigma^{(l)}\left(W^{(l)} h^{(l-1)}(x) + b^{(l)}\right)
$$

Final prediction:

$$
f(x) = W^{(L)} h^{(L-1)}(x)
$$

We define learned feature map:

$$
\phi_{NN}(x) := h^{(L-1)}(x)
$$

Neural kernel:

$$
k(x_i, x_j) = \phi_{NN}(x_i)^\top \phi_{NN}(x_j)
$$

---

### **Implementation Steps**

- Train MLP
- Extract features:
  
  $$
  \Phi = \{\phi_{NN}(x_i)\}_{i=1}^n
  $$

- Compute Gram matrix:

  $$
  K_{ij} = k(x_i, x_j)
  $$

- Train SVR using precomputed kernel.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

# Add path to mlp.py
sys.path.append(r"C:\Users\PUSHKAR\Desktop\DL-Assignments\Assignment-1\mlp")

from mlp import MLP

In [ ]:
# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_csv(r"C:\Users\PUSHKAR\Desktop\DL-Assignments\Assignment-1\iit_h_mess_dataset.csv")

X = df.drop(columns=["mess_duration"]).values
y = df["mess_duration"].values.reshape(-1, 1)

In [ ]:
# train / val / test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [ ]:
# feature scaling (CRITICAL)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled   = y_scaler.transform(y_val)
y_test_scaled  = y_scaler.transform(y_test)

In [ ]:
layer_sizes = [X_train.shape[1], 64, 64, 1]
activations = ["tanh", "tanh", "tanh"]

model = MLP(
    layer_sizes=layer_sizes,
    activations=activations,
    learning_rate=0.01
)

In [ ]:
def huber_loss(y_true, y_pred, delta=1.0):
    error = y_pred - y_true
    abs_error = np.abs(error)

    quadratic = np.minimum(abs_error, delta)
    linear = abs_error - quadratic

    return np.mean(0.5 * quadratic**2 + delta * linear)


def huber_grad(y_true, y_pred, delta=1.0):
    error = y_pred - y_true
    grad = np.where(np.abs(error) <= delta, error, delta * np.sign(error))
    return grad

In [ ]:
lambda_l2 = 0.01
epochs = 500

loss_history = []

for _ in range(epochs):

    # Forward
    y_pred = model.forward(X_train)

    # Compute Huber loss
    loss = huber_loss(y_train_scaled.T, y_pred)
    loss_history.append(loss)

    # Compute gradient of loss wrt output
    dA = huber_grad(y_train_scaled.T, y_pred)

    grads_W = {}
    grads_b = {}

    m = X_train.shape[0]

    # Backprop
    for l in reversed(range(1, model.L + 1)):
        dZ = dA * model.act_deriv[l](model.Z[l])
        grads_W[l] = (1/m) * dZ @ model.A[l-1].T
        grads_b[l] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

        # L2 regularization
        grads_W[l] += lambda_l2 * model.W[l]

        dA = model.W[l].T @ dZ

    # Update
    for l in range(1, model.L + 1):
        model.W[l] -= model.optimizer.lr * grads_W[l]
        model.b[l] -= model.optimizer.lr * grads_b[l]

In [ ]:
def extract_features(model, X):
    model.forward(X)
    L_minus_1 = model.L - 1
    return model.A[L_minus_1].T

Phi_train = extract_features(model, X_train)
Phi_test  = extract_features(model, X_test)

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
Phi_2d = tsne.fit_transform(Phi_train)

plt.figure()
plt.scatter(Phi_2d[:,0], Phi_2d[:,1], c=y_train.flatten(), s=20)
plt.title("t-SNE of NN Features")
plt.show()

In [ ]:
def neural_kernel_matrix(Phi_A, Phi_B):
    return Phi_A @ Phi_B.T

K_train = neural_kernel_matrix(Phi_train, Phi_train)
K_test  = neural_kernel_matrix(Phi_test, Phi_train)

In [ ]:
svr = SVR(kernel='precomputed', C=1.0, epsilon=0.1)
svr.fit(K_train, y_train_scaled.ravel())

y_pred_scaled = svr.predict(K_test)

# Inverse scaling
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1,1))

mse = mean_squared_error(y_test, y_pred)
print("Neural Kernel SVR Test MSE:", mse)

## 7.2.2 Comparison with Standard Kernels

We compare the following kernels for Support Vector Regression (SVR):

### Kernels

**(a) Linear Kernel**
$$
k(x_i,x_j) = x_i^T x_j
$$

**(b) Polynomial Kernel**
$$
k(x_i,x_j) = (x_i^T x_j + c)^d, \quad d \in \{2,3\}
$$

**(c) RBF (Gaussian) Kernel**
$$
k(x_i,x_j) = \exp(-\gamma \|x_i - x_j\|^2),
\quad \gamma \in \{0.01, 0.1, 1.0\}
$$

**(d) Neural Kernel**
$$
k(x_i,x_j) = \phi_{NN}(x_i)^T \phi_{NN}(x_j)
$$

where
$$
\phi_{NN}(x) = h^{(L-1)}(x)
$$

---

### For each kernel:

- Report test MSE
- Plot 2D decision surface (via PCA projection)
- Visualize kernel matrix:
  $$
  K_{ij} = k(x_i,x_j)
  $$

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import rbf_kernel

In [ ]:
def linear_kernel_matrix(XA, XB):
    return XA @ XB.T

def polynomial_kernel_matrix(XA, XB, degree=2, c=1):
    return (XA @ XB.T + c) ** degree

def rbf_kernel_matrix(XA, XB, gamma=0.1):
    return rbf_kernel(XA, XB, gamma=gamma)

def neural_kernel_matrix(PhiA, PhiB):
    return PhiA @ PhiB.T

In [ ]:
Phi_train = extract_features(model, X_train)
Phi_test  = extract_features(model, X_test)

In [ ]:
def train_evaluate_kernel(K_train, K_test, y_train, y_test, name):

    svr = SVR(kernel='precomputed', C=1.0, epsilon=0.1)
    svr.fit(K_train, y_train.ravel())

    y_pred_scaled = svr.predict(K_test)
    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1,1))

    mse = mean_squared_error(y_test, y_pred)

    print(f"{name} Test MSE: {mse:.4f}")

    return mse

In [ ]:
results = {}

In [ ]:
# Linear
K_train = linear_kernel_matrix(X_train, X_train)
K_test  = linear_kernel_matrix(X_test, X_train)

results["Linear"] = train_evaluate_kernel(
    K_train, K_test,
    y_train_scaled, y_test,
    "Linear Kernel"
)

In [ ]:
# Polynomial
for d in [2,3]:
    K_train = polynomial_kernel_matrix(X_train, X_train, degree=d)
    K_test  = polynomial_kernel_matrix(X_test, X_train, degree=d)

    results[f"Poly_d{d}"] = train_evaluate_kernel(
        K_train, K_test,
        y_train_scaled, y_test,
        f"Polynomial Kernel (d={d})"
    )

In [ ]:
# rbf
for gamma in [0.01, 0.1, 1.0]:
    K_train = rbf_kernel_matrix(X_train, X_train, gamma=gamma)
    K_test  = rbf_kernel_matrix(X_test, X_train, gamma=gamma)

    results[f"RBF_gamma{gamma}"] = train_evaluate_kernel(
        K_train, K_test,
        y_train_scaled, y_test,
        f"RBF Kernel (gamma={gamma})"
    )

In [ ]:
# neural
K_train = neural_kernel_matrix(Phi_train, Phi_train)
K_test  = neural_kernel_matrix(Phi_test, Phi_train)

results["Neural Kernel"] = train_evaluate_kernel(
    K_train, K_test,
    y_train_scaled, y_test,
    "Neural Kernel"
)

In [ ]:
def plot_kernel_matrix(K, title):
    plt.figure()
    plt.imshow(K, aspect='auto')
    plt.colorbar()
    plt.title(title)
    plt.show()

In [ ]:
plot_kernel_matrix(K_train, "Neural Kernel Matrix")

In [ ]:
def plot_decision_surface(X, y, kernel_func, name):

    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)

    x_min, x_max = X_2d[:,0].min()-1, X_2d[:,0].max()+1
    y_min, y_max = X_2d[:,1].min()-1, X_2d[:,1].max()+1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_orig = pca.inverse_transform(grid)

    K_grid = kernel_func(grid_orig, X_train)

    svr = SVR(kernel='precomputed')
    K_train = kernel_func(X_train, X_train)
    svr.fit(K_train, y_train_scaled.ravel())

    Z = svr.predict(K_grid)
    Z = Z.reshape(xx.shape)

    plt.figure()
    plt.contourf(xx, yy, Z, levels=50)
    plt.scatter(X_2d[:,0], X_2d[:,1], c=y.flatten(), edgecolors='k')
    plt.title(name)
    plt.show()

In [ ]:
plot_decision_surface(X_train, y_train,
                      lambda A,B: rbf_kernel_matrix(A,B,gamma=0.1),
                      "RBF Decision Surface")

In [ ]:
print("\nFinal Comparison:")
for k,v in results.items():
    print(f"{k}: {v:.4f}")